# Week 01 · Day 01 — LLM Fundamentals

**Topics:** Tokenization · Context windows · Temperature & sampling

Run cells top-to-bottom. Add your API keys as Colab Secrets or set them in the cell below.

In [ ]:
%pip install -q openai tiktoken

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1 · Tokenization

LLMs don't see characters or words — they see **tokens**. A token is roughly 4 characters or ¾ of a word in English. `tiktoken` is OpenAI's tokenizer library.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

text = "Large language models tokenize text differently than humans read it."
tokens = enc.encode(text)

print(f"Text:   {text}")
print(f"Tokens: {tokens}")
print(f"Count:  {len(tokens)} tokens")
print()

# Decode each token individually to see the splits
for tok in tokens:
    decoded = enc.decode([tok])
    print(f"  {tok:6d} → {repr(decoded)}")

In [ ]:
# Tokenization varies by model family
samples = [
    "Hello, world!",
    "ChatGPT",
    "🚀",
    "   leading spaces",
    "1234567890",
    "supercalifragilisticexpialidocious",
]

for s in samples:
    n = len(enc.encode(s))
    print(f"{n:3d} tokens — {repr(s)}")

## 2 · Context Windows

The context window is the maximum number of tokens the model can process in one call — both input and output combined. Exceeding it causes an error; approaching it affects coherence.

In [ ]:
# Context limits by model (as of 2025)
context_windows = {
    "gpt-4o":               128_000,
    "gpt-4o-mini":          128_000,
    "claude-sonnet-4-5":    200_000,
    "claude-haiku-4-5":     200_000,
    "gemini-1.5-pro":     1_000_000,
    "llama-3.1-8b":         128_000,
}

print(f"{'Model':<25} {'Context window':>15} {'Approx pages':>15}")
print("-" * 58)
for model, tokens in context_windows.items():
    pages = tokens // 500  # ~500 tokens/page
    print(f"{model:<25} {tokens:>15,} {pages:>14,}p")

In [ ]:
# Count tokens before sending to avoid errors
def count_tokens(messages: list[dict], model: str = "gpt-4o") -> int:
    enc = tiktoken.encoding_for_model(model)
    total = 0
    for msg in messages:
        total += 4  # per-message overhead
        for val in msg.values():
            total += len(enc.encode(str(val)))
    total += 2  # reply primer
    return total

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain transformers in one paragraph."},
]
print(f"Estimated prompt tokens: {count_tokens(messages)}")

## 3 · Temperature and Sampling

**Temperature** controls randomness. `temperature=0` is deterministic (greedy decoding). Higher values increase diversity but reduce reliability.

| Task | Recommended temperature |
|------|------------------------|
| Classification, extraction | 0.0 |
| Q&A, summarization | 0.2–0.4 |
| Chatbots | 0.7 |
| Creative writing | 1.0–1.2 |

In [ ]:
prompt = "Write a one-sentence tagline for a coffee shop."

for temp in [0.0, 0.7, 1.2]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=60,
    )
    print(f"temp={temp}: {response.choices[0].message.content.strip()}")

In [ ]:
# Temperature=0 is deterministic — run 3 times, get same result
results = []
for _ in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "What is 17 × 23?"}],
        temperature=0,
        max_tokens=20,
    )
    results.append(r.choices[0].message.content.strip())

print("Three runs at temperature=0:")
for i, r in enumerate(results, 1):
    print(f"  Run {i}: {r}")
print(f"All identical: {len(set(results)) == 1}")

## 4 · Top-p (Nucleus Sampling)

**top_p** restricts the token pool to the smallest set of tokens whose cumulative probability ≥ p. `top_p=0.1` picks only high-probability tokens; `top_p=1.0` considers everything.

In practice: use **temperature OR top_p**, not both. OpenAI recommends altering one and leaving the other at its default.

In [ ]:
prompt = "Continue this story: The robot opened its eyes for the first time and"

for top_p in [0.1, 0.5, 1.0]:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        top_p=top_p,
        max_tokens=50,
    )
    text = r.choices[0].message.content.strip()
    print(f"top_p={top_p}: {text}")
    print()

## 5 · Token Cost Estimation

Always estimate cost before running large batches.

In [ ]:
# Pricing per 1M tokens (USD, as of 2025 — verify at platform.openai.com/pricing)
PRICING = {
    "gpt-4o":       {"input": 2.50, "output": 10.00},
    "gpt-4o-mini":  {"input": 0.15, "output": 0.60},
}

def estimate_cost(input_tokens: int, output_tokens: int, model: str = "gpt-4o") -> float:
    p = PRICING[model]
    return (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000

# Scenario: 1000 customer support queries, avg 500 input + 200 output tokens
for model in PRICING:
    cost = estimate_cost(500, 200, model) * 1000
    print(f"{model:<15} 1k queries → ${cost:.2f}")

## Summary

- Tokens ≠ words. Use `tiktoken` to count before sending.
- Context window = total tokens in + out. Exceeding it raises an error.
- `temperature=0` for deterministic tasks; 0.7–1.0 for creative tasks.
- Use `top_p` **or** `temperature` — not both.
- Always estimate cost before batch jobs.

**Next:** [Prompt Engineering](week01-day01-prompt-engineering.ipynb)